# Getting started with GEDI L2A data in Python

This notebook introduces the **GEDI Level 2A Elevation and Height Metrics** product
([GEDI02_A.002](https://doi.org/10.5067/GEDI/GEDI02_A.002)). You will learn how to

1. download GEDI data and find your way around the HDF5 file structure,
2. turn GEDI laser shots into a GeoDataFrame and clip them to a study area,
3. read and plot the **relative height (RH) metrics** of a single shot,
4. quality-filter the data and plot a beam transect,
5. map canopy height and export the result for use in a GIS.

**How to use this notebook:** run the cells from top to bottom with `Shift + Enter`.
Nothing needs to be installed. On Binder your session is temporary. If you want to keep your work,
use *File → Download* before you close the tab.

> Adapted from the LP DAAC tutorial *Getting Started with GEDI L2A Version 2 Data in Python*
> ([nasa/GEDI-Data-Resources](https://github.com/nasa/GEDI-Data-Resources), Apache License 2.0).
> See `NOTICE` in the repository for details.

## 1. Setup

In [ ]:
from pathlib import Path
import urllib.request

import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

DATA = Path("../data")
OUT = Path("../output")
OUT.mkdir(exist_ok=True)

## 1.1 Download the data

GEDI data are distributed by NASA's LP DAAC. A full GEDI file (*granule*) covers a quarter of an orbit
around the Earth and is 1–3 GB. Downloading from NASA requires a free Earthdata account.

Your instructor has already downloaded GEDI files covering the Kalkalpen region and put them
on a file share, so you can download them without an account. This can take a few minutes,
so start the cell below right away:

In [ ]:
import zipfile

# Download links provided by your instructor: single .h5 files or one .zip containing .h5 files
DATA_URLS = [
    "https://zenodo.org/records/XXXXXXX/files/gedi_l2a_kalkalpen.zip?download=1",
]

def download(url):
    name = url.split("/")[-1].split("?")[0]
    target = DATA / name
    if not target.exists():
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(url, target)
        print(f"  done ({target.stat().st_size / 1e6:.0f} MB)")
    if target.suffix == ".zip":                  # unpack .h5 files, then delete the zip
        with zipfile.ZipFile(target) as z:
            for member in z.namelist():
                if member.endswith(".h5") and not (DATA / Path(member).name).exists():
                    (DATA / Path(member).name).write_bytes(z.read(member))
        target.unlink()

for url in DATA_URLS:
    download(url)

For reference, this is how the original files are downloaded from NASA with the `earthaccess` package
(needs an Earthdata login, so **don't run this here**):

```python
import earthaccess
earthaccess.login()
results = earthaccess.search_data(
    short_name="GEDI02_A", version="002",
    bounding_box=(14.25, 47.68, 14.62, 47.88),         # west, south, east, north
    temporal=("2022-05-01", "2023-09-30"))
earthaccess.download(results[:1], local_path="raw/")  # ~1-3 GB per granule!
```

In [ ]:
files = sorted(DATA.glob("*GEDI02_A*.h5"))
assert files, "No GEDI files in data/ - run the download cell above first."
for i, f in enumerate(files):
    print(f"{i:>2}  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

GEDI file names follow a fixed pattern, e.g. `GEDI02_A_2019170155833_O02932_02_T02267_02_003_01_V002.h5`:

| Part | Meaning |
|---|---|
| `GEDI02_A` | product short name |
| `2019170155833` | acquisition date and time (YYYY DDD HHMMSS, DDD = day of year) |
| `O02932` | orbit number |
| `02` | sub-orbit granule (1–4) |
| `T02267` | track number |
| `02` | positioning/pointing type (02 or higher = final) |
| `003` / `01` | processing software version / granule production version |
| `V002` | product version |

**Question:** On which date was the first file acquired?

## 2. Explore the HDF5 file

HDF5 files are organised like a file system: **groups** (folders) contain **datasets** (arrays) and **attributes** (metadata).

In [ ]:
FILE_NO = 0                      # pick a file from the list above (try a summer acquisition!)
gedi = h5py.File(files[FILE_NO], "r")
list(gedi.keys())

In [ ]:
# File-level metadata
meta = gedi["METADATA/DatasetIdentification"].attrs
for k in meta:
    print(f"{k}: {meta[k]}")

GEDI has 3 lasers. One laser is split into two **coverage beams**, and the other two lasers are **full power beams**.
Each of these four beams is dithered across-track to produce two ground tracks. This gives **8 beam tracks**
(`BEAM0000` … `BEAM1011`). Full power beams penetrate dense canopy better.

In [ ]:
beams = [k for k in gedi.keys() if k.startswith("BEAM")]
for b in beams:
    print(f"{b}: {gedi[b].attrs['description']:<20} {gedi[b]['shot_number'].shape[0]:>6} shots")

In [ ]:
# What is inside one beam group? (print the tree)
def show(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"{name:<45} {str(obj.shape):<12} {obj.dtype}")

biggest = max(beams, key=lambda b: gedi[b]["shot_number"].shape[0])
print(biggest)
gedi[biggest].visititems(show)

In [ ]:
# Each dataset carries a description attribute
for name in ["rh", "elev_lowestmode", "elev_highestreturn", "quality_flag", "sensitivity", "degrade_flag"]:
    print(f"{name}: {gedi[biggest][name].attrs.get('description', '')}\n")

## 3. From laser shots to a GeoDataFrame

Every GEDI **shot** is one laser footprint (~25 m diameter) on the ground. It has a unique `shot_number`.
`lat_lowestmode` and `lon_lowestmode` give the location of the lowest detected mode (the ground).

The function below reads the most important datasets of one beam into a table.
`rh` has shape *(shots × 101)*, i.e. the relative heights at 0, 1, 2 … 100 % of returned energy.

In [ ]:
def read_beam(h5, beam):
    g = h5[beam]
    rh = g["rh"][()]
    df = pd.DataFrame({
        "beam": beam,
        "beam_type": g.attrs["description"],
        "idx": np.arange(rh.shape[0]),                 # position inside the beam (to find the shot again)
        "shot_number": g["shot_number"][()].astype(str),
        "lat": g["lat_lowestmode"][()],
        "lon": g["lon_lowestmode"][()],
        "elev_ground": g["elev_lowestmode"][()],
        "elev_top": g["elev_highestreturn"][()],
        "dem_tandemx": g["digital_elevation_model"][()],
        "rh25": rh[:, 25], "rh50": rh[:, 50], "rh75": rh[:, 75],
        "rh98": rh[:, 98], "rh100": rh[:, 100],
        "quality_flag": g["quality_flag"][()],
        "degrade_flag": g["degrade_flag"][()],
        "sensitivity": g["sensitivity"][()],
        "selected_algorithm": g["selected_algorithm"][()],
    })
    return df

df = pd.concat([read_beam(gedi, b) for b in beams if gedi[b]["shot_number"].shape[0] > 0],
               ignore_index=True)
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")
print(f"{len(gdf)} shots in the region")
gdf.head()

In [ ]:
aoi = gpd.read_file(DATA / "aoi.geojson")       # our study area: Kalkalpen National Park

fig, ax = plt.subplots(figsize=(8, 6))
gdf.plot(ax=ax, column="beam", markersize=0.5, legend=True)
aoi.boundary.plot(ax=ax, color="red", lw=2)
ax.set(xlabel="Longitude", ylabel="Latitude", title="GEDI shots in the region and the study area (red)")
plt.show()

### 3.1 Clip to the study area

The file covers a whole region, but we only want the shots inside the study area.
`GeoDataFrame.clip()` keeps only the geometries inside a polygon.

> With full-size granules (1–3 GB) you would clip *before* reading everything: read only `lat_lowestmode` and
> `lon_lowestmode`, build a boolean mask, and then read only the masked rows of the large datasets (like `rh`).
> See `prep/prepare_subset.py` for an example.

In [ ]:
gdf_region = gdf
gdf = gdf_region.clip(aoi)
print(f"{len(gdf)} of {len(gdf_region)} shots are inside the study area")

m = aoi.explore(style_kwds={"fill": False, "color": "red"})
gdf.explore(m=m, column="beam", tooltip=["shot_number", "beam_type", "quality_flag"],
            marker_kwds={"radius": 2})

Zoom in: the shots are arranged in parallel tracks, one per beam.

**Question:** Roughly how far apart are neighbouring shots along a track, and how far apart are the tracks?

## 4. Relative height metrics of a single shot

**How to read RH metrics:**
- `rh` gives the height (m) **above the ground** (`elev_lowestmode`) below which *x* % of the returned waveform energy lies.
- **RH100** = `elev_highestreturn` − `elev_lowestmode`, the top of the canopy as seen by the lidar.
- Low RH values (RH1–RH20) are often **negative**. Part of the ground return lies below the centre of the ground peak.
- RH metrics are meant for vegetated surfaces. Over bare ground or water they can be confusing.

First, pick a good-quality shot in tall forest (here: a shot close to the 90th percentile of RH98).

In [ ]:
good = gdf[(gdf.quality_flag == 1) & (gdf.degrade_flag == 0)]
target = good.rh98.quantile(0.9)
shot = good.loc[(good.rh98 - target).abs().idxmin()]
shot[["shot_number", "beam", "idx", "lat", "lon", "elev_ground", "rh98", "selected_algorithm"]]

In [ ]:
g = gedi[shot.beam]
rh_shot = g["rh"][shot.idx]                 # 101 values, metres above ground
z_ground = g["elev_lowestmode"][shot.idx]
z_top = g["elev_highestreturn"][shot.idx]
rh_elev = rh_shot + z_ground                # convert heights to elevations

pct = np.arange(101)
fig, ax = plt.subplots(figsize=(5, 7))
ax.plot(pct, rh_elev, color="black", lw=2.5, label=f"RH profile (algorithm {shot.selected_algorithm})")
ax.axhline(z_ground, color="saddlebrown", label="Ground (elev_lowestmode)")
ax.axhline(z_top, color="navy", label="RH100 (elev_highestreturn)")
for p, c in [(25, "skyblue"), (50, "tab:blue"), (75, "tab:purple")]:
    ax.axhline(rh_elev[p], color=c, ls="--", label=f"RH{p}")
ax.set(xlabel="Percent energy returned", ylabel="Elevation (m)", xlim=(0, 100),
       title=f"Shot {shot.shot_number}")
ax.legend(loc="lower right", fontsize=8)
plt.show()

**Questions:**
- How tall is the canopy at this shot?
- In which height range is most of the canopy material? (Where is the curve steepest?)

### 4.1 Different algorithm setting groups

GEDI processes every waveform with **six algorithm setting groups** (a1–a6). The values in the beam root
come from the algorithm selected for that shot (`selected_algorithm`). The results of all six are stored in
the `geolocation` subgroup. Note: `rh_a<n>` is stored in **centimetres**.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 7))
for a in range(1, 7):
    rh_a = g[f"geolocation/rh_a{a}"][shot.idx] / 100          # cm -> m
    z_a = g[f"geolocation/elev_lowestmode_a{a}"][shot.idx]
    lw = 3 if a == shot.selected_algorithm else 1
    ax.plot(pct, rh_a + z_a, lw=lw, label=f"a{a}" + (" (selected)" if a == shot.selected_algorithm else ""))
ax.set(xlabel="Percent energy returned", ylabel="Elevation (m)", xlim=(0, 100),
       title="RH profile by algorithm setting group")
ax.legend(fontsize=8)
plt.show()

**Question:** Where do the algorithms differ most, near the ground or in the canopy? Why could that be?

## 5. Beam transect and quality filtering

Let's look at one full power beam along its whole track inside our area.

In [ ]:
counts = gdf[gdf.beam_type.str.lower().str.contains("full power")].beam.value_counts()
beam = counts.index[0]   # the full power beam with most shots in the study area
tr = gdf[gdf.beam == beam].sort_values("lat")

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(tr.lat, tr.elev_ground, s=3, color="saddlebrown", label="GEDI ground")
ax.scatter(tr.lat, tr.elev_top, s=3, color="green", label="GEDI canopy top")
ax.plot(tr.lat, tr.dem_tandemx, color="grey", lw=1, label="TanDEM-X DEM")
ax.set(xlabel="Latitude", ylabel="Elevation (m)", title=f"{beam}: all shots (unfiltered)")
ax.legend()
plt.show()

That looks messy: clouds and weak returns produce elevations far from the terrain. GEDI provides several
datasets for filtering:

| Dataset | Keep shots where | Why |
|---|---|---|
| `quality_flag` | `== 1` | shot meets criteria for energy, sensitivity, amplitude and surface tracking |
| `degrade_flag` | `== 0` | shot was not acquired during a period of degraded pointing/positioning |
| `sensitivity` | `> 0.95` | maximum canopy cover the shot can penetrate. `quality_flag` already uses 0.9 over land, but the GEDI user guide recommends higher thresholds for dense forest |

In [ ]:
def quality_filter(d, min_sensitivity=0.95):
    return d[(d.quality_flag == 1) & (d.degrade_flag == 0) & (d.sensitivity > min_sensitivity)]

tr_f = quality_filter(tr)
print(f"{len(tr_f)} of {len(tr)} shots kept ({len(tr_f) / len(tr):.0%})")

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(tr_f.lat, tr_f.elev_ground, s=4, color="saddlebrown", label="GEDI ground")
ax.scatter(tr_f.lat, tr_f.elev_top, s=4, color="green", label="GEDI canopy top")
ax.plot(tr.lat, tr.dem_tandemx, color="grey", lw=1, label="TanDEM-X DEM")
ax.set(xlabel="Latitude", ylabel="Elevation (m)", title=f"{beam}: quality-filtered")
ax.legend()
plt.show()

### 5.1 Extra check for mountain terrain

In steep terrain, GEDI sometimes picks the wrong mode as the ground. A simple extra check compares the GEDI
ground elevation with an independent DEM (TanDEM-X is included in the file) and drops shots with large differences.
The threshold is a choice. Try different values!

In [ ]:
gdf_f = quality_filter(gdf).copy()
gdf_f["dz"] = gdf_f.elev_ground - gdf_f.dem_tandemx

gdf_f.dz.clip(-100, 100).hist(bins=80, figsize=(7, 3))
plt.xlabel("GEDI ground − TanDEM-X (m)"); plt.ylabel("Shots"); plt.show()

MAX_DZ = 50   # metres
gdf_f = gdf_f[gdf_f.dz.abs() < MAX_DZ]
print(f"{len(gdf_f)} shots after all filters (of {len(gdf)})")

## 6. Map canopy height

In [ ]:
m = aoi.explore(style_kwds={"fill": False, "color": "red"})
gdf_f.explore(m=m, column="rh98", cmap="viridis", vmin=0, vmax=40,
              tooltip=["shot_number", "beam", "rh98", "elev_ground", "sensitivity"],
              marker_kwds={"radius": 3})

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
gdf_f.rh98.hist(bins=40, ax=axs[0])
axs[0].set(xlabel="RH98 (m)", ylabel="Shots", title="Canopy height distribution")
axs[1].scatter(gdf_f.elev_ground, gdf_f.rh98, s=4, alpha=0.5)
axs[1].set(xlabel="Ground elevation (m a.s.l.)", ylabel="RH98 (m)", title="Canopy height vs. elevation")
plt.tight_layout(); plt.show()

**Questions:**
- How does canopy height change with elevation? Can you spot the treeline?
- Why do we use RH98 instead of RH100 as "canopy height"?

## 7. Export for GIS

In [ ]:
cols = ["shot_number", "beam", "beam_type", "elev_ground", "elev_top", "rh25", "rh50", "rh75",
        "rh98", "rh100", "sensitivity", "selected_algorithm", "dz", "geometry"]
gdf_f[cols].to_file(OUT / "gedi_l2a_filtered.gpkg", driver="GPKG")
gdf_f[cols].to_file(OUT / "gedi_l2a_filtered.geojson", driver="GeoJSON")
print(list(OUT.iterdir()))

The files are in the `output/` folder (see the file browser on the left). Right-click a file and choose **Download**
to open it in QGIS.

In [ ]:
gedi.close()

## Exercises

1. Repeat the transect plot (Section 5) for a **coverage beam**. How many shots survive the quality filter compared to a full power beam?
2. Change the sensitivity threshold to 0.9 and 0.98. How do the number of shots and the canopy height distribution change?
3. Change `FILE_NO` in Section 2 to another file and run the notebook again. Do you see differences between the acquisitions (season, time of day: `solar_elevation`)?
4. Add `land_cover_data/landsat_treecover` to `read_beam()` and plot it against RH98.
5. Draw your own study area (e.g. on [geojson.io](https://geojson.io)), save it as GeoJSON in `data/`, and clip to it instead.

## Further reading
- GEDI L2A product page and user guide: https://doi.org/10.5067/GEDI/GEDI02_A.002
- LP DAAC GEDI tutorials (L1B waveforms, L2B canopy cover): https://github.com/nasa/GEDI-Data-Resources
- GEDI mission: https://gedi.umd.edu